# Transfer Learning & Fine-Tuning Notebook

> Hands-on Build It and Exercises.

## Build It

### Step 1: Load a pretrained backbone and inspect it

In [ ]:
```python

import torch

import torch.nn as nn

from torchvision.models import resnet18, ResNet18_Weights

backbone = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)

print(backbone)

print()

print("classifier head:", backbone.fc)

print("feature dim:", backbone.fc.in_features)

In [ ]:
```

`ResNet18` has four stages (`layer1..layer4`) plus a stem and a `fc` head. Every torchvision classification backbone has an analogous structure.

### Step 2: Feature extraction — freeze everything, replace the head

In [ ]:
```python

def make_feature_extractor(num_classes=10):

    model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)

    for p in model.parameters():

        p.requires_grad = False

    model.fc = nn.Linear(model.fc.in_features, num_classes)

    return model

model = make_feature_extractor(num_classes=10)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

frozen = sum(p.numel() for p in model.parameters() if not p.requires_grad)

print(f"trainable: {trainable:>10,}")

print(f"frozen:    {frozen:>10,}")

In [ ]:
```

Only `model.fc` is trainable. The backbone is a frozen feature extractor.

### Step 3: Discriminative fine-tuning

A utility that builds parameter groups with stage-specific learning rates.

In [ ]:
```python

def discriminative_param_groups(model, base_lr=1e-3, decay=0.3):

    stages = [

        ["conv1", "bn1"],

        ["layer1"],

        ["layer2"],

        ["layer3"],

        ["layer4"],

        ["fc"],

    ]

    groups = []

    for i, names in enumerate(stages):

        lr = base_lr * (decay ** (len(stages) - 1 - i))

        params = [p for n, p in model.named_parameters()

                  if any(n.startswith(k) for k in names)]

        if params:

            groups.append({"params": params, "lr": lr, "name": "_".join(names)})

    return groups

model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)

model.fc = nn.Linear(model.fc.in_features, 10)

for p in model.parameters():

    p.requires_grad = True

groups = discriminative_param_groups(model)

for g in groups:

    print(f"{g['name']:>10s}  lr={g['lr']:.2e}  params={sum(p.numel() for p in g['params']):>8,}")

In [ ]:
```

`decay=0.3` means each stage trains at 30% of the rate of the next one. `fc` gets `base_lr`, `layer4` gets `0.3 * base_lr`, `conv1` gets `0.3^5 * base_lr ≈ 0.00243 * base_lr`. Extreme sounding; empirically it works.

### Step 4: BatchNorm handling

Helper to freeze BN running statistics without freezing its weights.

In [ ]:
```python

def freeze_bn_stats(model):

    for m in model.modules():

        if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d)):

            m.eval()

            for p in m.parameters():

                p.requires_grad = False

    return model

In [ ]:
```

Call it after you set `model.train()` at the start of every epoch. `model.train()` flips everything to training mode; this reverses it only for BN layers.

### Step 5: A minimal end-to-end fine-tuning loop

In [ ]:
```python

from torch.optim import SGD

from torch.utils.data import DataLoader

from torch.optim.lr_scheduler import CosineAnnealingLR

import torch.nn.functional as F

def fine_tune(model, train_loader, val_loader, device, epochs=5, base_lr=1e-3, freeze_bn=False):

    model = model.to(device)

    groups = discriminative_param_groups(model, base_lr=base_lr)

    optimizer = SGD(groups, momentum=0.9, weight_decay=1e-4, nesterov=True)

    scheduler = CosineAnnealingLR(optimizer, T_max=epochs)

    for epoch in range(epochs):

        model.train()

        if freeze_bn:

            freeze_bn_stats(model)

        tr_loss, tr_correct, tr_total = 0.0, 0, 0

        for x, y in train_loader:

            x, y = x.to(device), y.to(device)

            logits = model(x)

            loss = F.cross_entropy(logits, y, label_smoothing=0.1)

            optimizer.zero_grad()

            loss.backward()

            optimizer.step()

            tr_loss += loss.item() * x.size(0)

            tr_total += x.size(0)

            tr_correct += (logits.argmax(-1) == y).sum().item()

        scheduler.step()

        model.eval()

        va_total, va_correct = 0, 0

        with torch.no_grad():

            for x, y in val_loader:

                x, y = x.to(device), y.to(device)

                pred = model(x).argmax(-1)

                va_total += x.size(0)

                va_correct += (pred == y).sum().item()

        print(f"epoch {epoch}  train {tr_loss/tr_total:.3f}/{tr_correct/tr_total:.3f}  "

              f"val {va_correct/va_total:.3f}")

    return model

In [ ]:
```

Five epochs with the above recipe on CIFAR-10 takes `ResNet18-IMAGENET1K_V1` from ~70% zero-shot linear-probe accuracy to ~93% fine-tuned accuracy. The head alone would plateau around 86% without ever touching the backbone.

### Step 6: Progressive unfreezing

A schedule that unfreezes one stage per epoch from the end toward the beginning. Mitigates feature drift at the cost of some extra epochs.

In [ ]:
```python

def progressive_unfreeze_schedule(model):

    stages = ["layer4", "layer3", "layer2", "layer1"]

    yielded = set()

    def start():

        for p in model.parameters():

            p.requires_grad = False

        for p in model.fc.parameters():

            p.requires_grad = True

    def unfreeze(epoch):

        if epoch < len(stages):

            name = stages[epoch]

            yielded.add(name)

            for n, p in model.named_parameters():

                if n.startswith(name):

                    p.requires_grad = True

            return name

        return None

    return start, unfreeze

In [ ]:
```

Call `start()` once before the first epoch. Call `unfreeze(epoch)` at the start of each epoch. Rebuild the optimizer whenever the set of trainable parameters changes, otherwise the frozen params still hold cached moments that confuse it.

## Exercises

In [ ]:
1. **(Easy)** Train a `ResNet18` as a linear probe (backbone frozen) and as a full fine-tune on the same synthetic-CIFAR dataset. Report both accuracies side by side. Explain which gap tells you the features transfer well and which tells you they do not.
2. **(Medium)** Introduce a bug on purpose: set `base_lr = 1e-1` on the backbone stage instead of the head. Show the training loss explode, then recover by applying the `discriminative_param_groups` helper. Record the LR at which each stage starts diverging.
3. **(Hard)** Take a medical imaging dataset (e.g. CheXpert-small, PatchCamelyon, or HAM10000) and compare three regimes: (a) ImageNet-pretrained frozen backbone + linear head; (b) ImageNet-pretrained fine-tune end-to-end; (c) scratch training. Report accuracy and compute cost for each. At what dataset size does scratch training become competitive?